# Registro histórico: 14_dermai_group_draft

Origem: `C56` — `Colab Notebooks/Cópia de DermAI_Melanoma_Classification.ipynb`.

Rascunho relacionado DermAI com contexto e funções de avaliação.

**Limitações conhecidas:** Notebook identifica Enzo Souza, Vinicius Hardy e Angelo Pereira. Metas de AUC/latência/tamanho são objetivos, não resultados demonstrados.

Cópia organizada em 25/10/2026. Código e texto originais preservados abaixo; saídas, contagens de execução e metadados do Colab removidos. As métricas salvas estão em `results/historical_evidence.json`. Não executar em lote: estes cadernos dependem do ambiente e de arquivos originais. Veja `docs/experiments.md` para o fluxo novo.


# DermAI: Classificação Binária de Melanoma em Imagens Dermatoscópicas

## Resumo do Projeto

Este projeto visa desenvolver uma ferramenta de **classificação binária (Melanoma: Sim/Não)** a partir de imagens dermatoscópicas de lesões de pele capturadas por smartphone com lente acoplada (microscópio/lente macro), facilitando a triagem rápida em ambientes de atenção primária.

O melanoma possui alta letalidade quando diagnosticado tardiamente. A sobrevida em 5 anos é de aproximadamente 100% se localizado, caindo drasticamente para cerca de 35% em estágios metastáticos (Fonte: SEER). Com cerca de 331,7 mil novos casos e 58,7 mil mortes anuais globalmente, a necessidade de triagem eficiente é urgente (Fonte: GLOBOCAN 2022).

**Objetivo Central:** Classificar Melanoma vs Não-melanoma em imagens dermatoscópicas com alta precisão e sensibilidade.

**Métricas de Validação (Alvo):**
- AUC ≥ 0,90
- Sensibilidade (Melanoma) ≥ 0,90
- Especificidade ≥ 0,75
- F1-Score (Classe Melanoma) e Balanced Accuracy

**Escopo e Restrições:** Protótipo com inferência on-device (<5MB, MobileNet/EfficientNet-Lite) e tempo de resposta < 300 ms. A ferramenta NÃO substitui o diagnóstico médico.

**Grupo:** Enzo Souza, Vinicius Hardy, Angelo Pereira


# 2. Introdução e Motivação

## Contexto Clínico

O melanoma é um tipo de câncer de pele altamente agressivo que, quando detectado precocemente, apresenta excelentes taxas de cura. No entanto, o diagnóstico tardio resulta em prognóstico desfavorável, com taxas de sobrevida significativamente reduzidas em estágios avançados.

## Problema de Saúde Pública

Globalmente, o melanoma representa um desafio crescente para os sistemas de saúde:
- **331.700 novos casos** diagnosticados anualmente
- **58.700 mortes** por ano em todo o mundo
- Taxa de sobrevida em 5 anos: **~100%** quando detectado precocemente vs **~35%** em estágios metastáticos

## Motivação Tecnológica

A aplicação de técnicas de **Visão Computacional** e **Deep Learning** pode revolucionar o processo de triagem dermatológica:

1. **Democratização do Acesso:** Permitir triagem em Unidades Básicas de Saúde (UBS) e teledermatologia
2. **Redução do Tempo de Espera:** Priorizar casos de alto risco para análise especializada
3. **Padronização:** Reduzir variabilidade na avaliação inicial de lesões suspeitas
4. **Escalabilidade:** Atender populações em áreas remotas ou com escassez de dermatologistas

## Aplicações Práticas

### Triagem em UBS e Teledermatologia
Permite a captura padronizada da lesão. O modelo pode priorizar imediatamente os casos de alto risco para análise de um dermatologista, reduzindo o tempo de espera por consultas especializadas.

### Monitoramento Domiciliar e Acompanhamento
Pacientes de risco ou aqueles com histórico de lesões podem monitorar sinais e nevos entre consultas. O sistema gera alertas baseados em mudanças significativas na imagem ao longo do tempo (crescimento, alteração de cor).

## Abordagem Técnica

Este projeto utiliza o dataset **HAM10000** (Harvested Adjusted Melanoma 10000), um benchmark reconhecido na dermatologia assistida por IA, para treinar modelos de classificação binária baseados em **Redes Neurais Convolucionais (CNNs)**.


# 3. Definição do Problema

## Formulação do Problema

Dado um conjunto de imagens dermatoscópicas de lesões de pele, o objetivo é desenvolver um modelo de **classificação binária** que possa distinguir entre:

- **Classe Positiva (1):** Melanoma
- **Classe Negativa (0):** Não-melanoma (todas as outras categorias histológicas)

## Desafios Principais

### 1. Desbalanceamento Severo de Classes

O dataset HAM10000 apresenta um desbalanceamento crítico na classificação binária:
- **Melanoma (Classe Positiva):** 1.113 imagens (~11% do total)
- **Não-Melanoma (Classe Negativa):** 8.902 imagens (~89% do total)

Este desequilíbrio representa um **risco significativo de viés** do modelo em favor da classe majoritária, comprometendo a Sensibilidade (capacidade de detectar melanomas verdadeiros).

### 2. Heterogeneidade da Qualidade da Imagem

O HAM10000, compilado de múltiplas fontes, apresenta variações que exigem robusto pré-processamento:

**Variação Intrínseca:**
- Imagens de diferentes dispositivos e técnicas levam a **iluminação, contraste e tonalidade de cor** heterogêneas

**Artefatos:**
- Frequente presença de **pelos** (obstrução) e **bolhas de gel** ou reflexos, que precisam ser mitigados por técnicas de remoção de ruído

**Padrão Comum:**
- Resolução típica de ~600×450 px após pré-processamento inicial
- Imagens são **recortadas e centralizadas** na lesão

### 3. Viés de Representação e Generalização

A maioria dos repositórios de dermatoscopia, incluindo o HAM10000, tem **baixa representação de tons de pele escuros**. É vital estar atento à questão de **fairness** e generalização.

### 4. Prevenção de Data Leakage

O dataset contém o campo **lesion_id**, indicando que **múltiplas imagens** podem ser da **mesma lesão**. Se não forem agrupadas, podemos ter a mesma lesão no treino e no teste, levando a resultados otimistas falsos (leakage).

## Estratégias de Mitigação

Para endereçar esses desafios, o projeto implementa:

1. **Class Weights ou Focal Loss** para compensar o desbalanceamento
2. **Técnicas de Oversampling/Augmentação** para aumentar a representação da classe minoritária
3. **Normalização de cor** (e.g., Shades-of-Gray), **remoção de pelos (inpainting)** e padronização de **resize/crop** para entrada da CNN (224×224)
4. **Split Estratificado** agrupado por lesion_id para garantir separação adequada entre treino/validação/teste

## Métricas de Sucesso

Dada a natureza crítica do problema (detecção de câncer), as métricas prioritárias são:

- **Sensibilidade (Recall) para Melanoma ≥ 0,90:** Minimizar falsos negativos
- **AUC-ROC ≥ 0,90:** Capacidade geral de discriminação
- **Especificidade ≥ 0,75:** Controlar falsos positivos
- **F1-Score (Classe Melanoma):** Equilíbrio entre precisão e recall
- **Balanced Accuracy:** Métrica que considera o desbalanceamento


# 4. Descrição dos Dados e Análise Exploratória (EDA)

## Dataset: HAM10000

O dataset **Harvested Adjusted Melanoma (HAM10000)** é o principal recurso para este projeto, sendo um benchmark reconhecido na dermatologia assistida por IA.

### Características do HAM10000

- **Tamanho:** 10.015 imagens dermatoscópicas em formato JPEG
- **Fonte:** ISIC Archive / Harvard Dataverse
- **Classes:** Originalmente 7 categorias histológicas (incluindo **mel** – Melanoma)
- **Vantagens:** Grande volume, público, inclui metadados clínicos (idade, sexo, localização) e é amplamente utilizado como referência
- **Licença:** CC BY-NC 4.0 (adequada para uso acadêmico/não comercial)

### Link para o Dataset

O dataset pode ser acessado através do [ISIC Archive](https://www.isic-archive.com/) ou [Harvard Dataverse](https://dataverse.harvard.edu/).

## Análise Exploratória de Dados (EDA)

### Distribuição de Classes (Classificação Binária)

Para o problema de classificação binária (Melanoma vs Não-melanoma), a distribuição é:

| Classe | Quantidade | Percentual |
|--------|-----------|-----------|
| **Melanoma (Classe Positiva)** | 1.113 imagens | ~11% do total |
| **Não-Melanoma (Classe Negativa)** | 8.902 imagens | ~89% do total |

Este desbalanceamento representa um **risco significativo de viés** do modelo em favor da classe majoritária, comprometendo a Sensibilidade.

### Metadados Multimodais

Os dados incluem **idade, sexo e localização anatômica** (p.ex., tronco, face). A combinação desses dados com a imagem (modelo multimodal) pode melhorar a performance preditiva.

### Heterogeneidade da Qualidade da Imagem

O HAM10000, compilado de múltiplas fontes, apresenta variações que exigem robusto pré-processamento:

- **Variação Intrínseca:** Iluminação, contraste e tonalidade de cor heterogêneas
- **Artefatos:** Presença de pelos (obstrução) e bolhas de gel ou reflexos
- **Padrão Comum:** Resolução típica de ~600×450 px após pré-processamento inicial

## Código Python: Exploração do Dataset


In [ ]:
# Importações e Configuração Inicial
import os
import random
import shutil
from pathlib import Path
from timeit import default_timer as timer

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
import requests

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.models as models
from torchvision import transforms, datasets
from torchvision.models import EfficientNet_B0_Weights
from sklearn.metrics import f1_score
import warnings
warnings.filterwarnings("ignore", message=".*worker.*")
warnings.filterwarnings("ignore", message=".*pid.*")
warnings.filterwarnings("ignore", message=".*fork.*")
warnings.filterwarnings("ignore", message=".*DataLoader.*")

# Configurar dispositivo
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Configurar seeds para reprodutibilidade
random.seed(91432)
np.random.seed(91432)
torch.manual_seed(91432)
if torch.cuda.is_available():
    torch.cuda.manual_seed(91432)

In [ ]:
# Configuração de Caminhos do Dataset
# NOTA: Ajuste os caminhos conforme sua estrutura local

# Para Google Colab (descomente se usar Colab):
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r "/content/drive/MyDrive/0-vgg" "/content/"

# Setup paths
image_path = Path("/content/0-vgg/base_dir")  # Ajuste conforme necessário
train_dir = image_path / "train_dir"
test_dir = image_path / "test_dir"
val_dir = image_path / "val_dir"

print(f"Dataset path: {image_path}")
print(f"Train directory: {train_dir}")
print(f"Test directory: {test_dir}")
print(f"Validation directory: {val_dir}")

In [ ]:
# Download de Funções Auxiliares
if Path("helper_functions.py").is_file():
    print("helper_functions.py already exists, skipping download")
else:
    print("Downloading helper_functions.py")
    request = requests.get("https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/main/helper_functions.py")
    with open("helper_functions.py", "wb") as f:
        f.write(request.content)

from helper_functions import accuracy_fn

In [ ]:
# Funções Customizadas para F1-Score e Early Stopping

def f1_score_melanoma(y_true: torch.Tensor, y_pred: torch.Tensor, class_names: list) -> float:
    """
    Calcula o F1-Score para a classe 'melanoma'.
    Assume que a classe 'melanoma' é a segunda classe (índice 1) na lista class_names.
    """
    y_true_np = y_true.cpu().numpy()
    y_pred_np = y_pred.cpu().numpy()

    try:
        f1 = f1_score(y_true_np, y_pred_np, average='binary', pos_label=1)
    except ValueError:
        f1 = 0.0

    return f1

class EarlyStoppingF1:
    """
    Early stops the training if validation F1-Score for 'melanoma' doesn't improve after a given patience.
    """
    def __init__(self, patience=10, verbose=False, delta=0, path='best_model_f1.pth', class_names=None):
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_f1_max = -float('inf')
        self.delta = delta
        self.path = path
        self.class_names = class_names

        if self.class_names and 'melanoma' in self.class_names:
            self.melanoma_index = self.class_names.index('melanoma')
        else:
            self.melanoma_index = 1

    def __call__(self, val_f1, model):
        score = val_f1

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_f1, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_f1, model)
            self.counter = 0

    def save_checkpoint(self, val_f1, model):
        if self.verbose:
            print(f'Validation F1-Score improved ({self.val_f1_max:.6f} --> {val_f1:.6f}). Saving model ...')
        torch.save(model.state_dict(), self.path)
        self.val_f1_max = val_f1

def calculate_val_metrics(model: torch.nn.Module,
                          data_loader: torch.utils.data.DataLoader,
                          loss_fn: torch.nn.Module,
                          device: torch.device,
                          class_names: list):
    """Calcula perda, acurácia e F1-Score para melanoma no conjunto de validação."""
    model.eval()
    val_loss, val_acc = 0, 0
    all_preds = []
    all_labels = []

    with torch.inference_mode():
        for X, y in data_loader:
            X, y = X.to(device, non_blocking=True), y.to(device, non_blocking=True)
            y_float = y.float().unsqueeze(1)

            logits = model(X)
            loss = loss_fn(logits, y_float)

            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).long()

            acc = accuracy_fn(y_true=y, y_pred=preds.squeeze(1))

            val_loss += loss.item()
            val_acc += acc

            all_preds.extend(preds.squeeze(1).cpu().tolist())
            all_labels.extend(y.cpu().tolist())

    val_loss /= len(data_loader)
    val_acc /= len(data_loader)

    all_labels_tensor = torch.tensor(all_labels)
    all_preds_tensor = torch.tensor(all_preds)

    val_f1 = f1_score_melanoma(all_labels_tensor, all_preds_tensor, class_names)

    return val_loss, val_acc, val_f1

In [ ]:
# Contagem de Imagens por Classe

def count_images_per_class(directory):
    """Counts the number of image files per class in a directory."""
    class_counts = {}
    for class_name in os.listdir(directory):
        class_dir = os.path.join(directory, class_name)
        if os.path.isdir(class_dir):
            image_count = 0
            for _, _, files in os.walk(class_dir):
                for file in files:
                    if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif')):
                        image_count += 1
            class_counts[class_name] = image_count
    return class_counts

# Contar imagens por classe em cada diretório
train_class_counts = count_images_per_class(train_dir)
test_class_counts = count_images_per_class(test_dir)
val_class_counts = count_images_per_class(val_dir)

print("\n=== Distribuição de Classes ===")
print(f"\nTrain: {train_class_counts}")
print(f"Test: {test_class_counts}")
print(f"Validation: {val_class_counts}")

# Visualização
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (title, counts) in zip(axes, [
    ("Train", train_class_counts),
    ("Test", test_class_counts),
    ("Validation", val_class_counts)
]):
    ax.bar(counts.keys(), counts.values(), color=['#2d5f5d', '#8ab8b6'])
    ax.set_title(title)
    ax.set_xlabel("Classe")
    ax.set_ylabel("Número de Imagens")
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Visualização de Amostras do Dataset

def visualize_samples(directory, num_samples=5):
    """Visualiza amostras aleatórias de cada classe."""
    classes = [d for d in os.listdir(directory) if os.path.isdir(os.path.join(directory, d))]

    fig, axes = plt.subplots(len(classes), num_samples, figsize=(15, 3*len(classes)))

    for i, class_name in enumerate(classes):
        class_dir = os.path.join(directory, class_name)
        images = [f for f in os.listdir(class_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        samples = random.sample(images, min(num_samples, len(images)))

        for j, img_name in enumerate(samples):
            img_path = os.path.join(class_dir, img_name)
            img = Image.open(img_path)

            if len(classes) > 1:
                axes[i, j].imshow(img)
                axes[i, j].axis('off')
                if j == 0:
                    axes[i, j].set_title(f"{class_name}", fontsize=12, fontweight='bold')
            else:
                axes[j].imshow(img)
                axes[j].axis('off')
                if j == 0:
                    axes[j].set_title(f"{class_name}", fontsize=12, fontweight='bold')

    plt.tight_layout()
    plt.show()

visualize_samples(train_dir, num_samples=5)

# 5. Metodologia e Arquitetura do Modelo

## 5.1 Pipeline de Processamento

O fluxo completo implementado no código segue quatro etapas principais:

1. **Carregamento das imagens**
   - As imagens dermatoscópicas são organizadas em pastas no padrão `ImageFolder`:
     - `train_dir/`
     - `val_dir/`
     - `test_dir/`
   - Cada pasta contém duas classes: **melanoma** e **não-melanoma**.

2. **Pré-processamento e Data Augmentation**
   - Redimensionamento, recorte, normalização e transformações aleatórias.
   - Aumenta a robustez do modelo em relação a variações de iluminação e enquadramento.

3. **Modelo CNN (EfficientNet-B0)**
   - Arquitetura pré-treinada no ImageNet.
   - Configurada para classificação binária com apenas **1 neurônio de saída**.

4. **Probabilidade e Decisão**
   - A saída do modelo é um **logit**.
   - Aplica-se a função `sigmoid` → probabilidade de melanoma.
   - Threshold = 0.5 para conversão em classe 0/1.


## 5.2 Organização do Dataset e DataLoaders

O código utiliza `torchvision.datasets.ImageFolder` para carregar automaticamente as imagens de:

- `train`
- `val`
- `test`

Para cada conjunto:

- Cria-se um objeto `Dataset`
- Define-se um `DataLoader` com batch size = 32
- `shuffle=True` apenas no treino
- `num_workers = 0` para compatibilidade com Google Colab
- `pin_memory=True` quando usando GPU

O código também contabiliza e imprime:

- Número de imagens por classe em cada partição
- Exibição de uma imagem aleatória com seu shape


In [ ]:
# Setup paths
image_path = Path("/content/balanced_ham10000-folder/content/ham10000_balanced_images")
train_dir = image_path / "train"
test_dir = image_path / "test"
val_dir = image_path / "val"
print(f"Dataset path: {image_path}")
print(f"Train directory: {train_dir}")
print(f"Test directory: {test_dir}")
print(f"Validation directory: {val_dir}")

## 5.3 Pré-processamento e Data Augmentation

O pipeline de transformações segue a normalização padrão do ImageNet.

### Transformações aplicadas no conjunto de treino

- `Resize((256, 256))`
- `RandomCrop(224)`
- `ColorJitter` (alterações aleatórias de iluminação e cor)
- `ToTensor()`
- `Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])`

Essas operações aumentam o conjunto efetivo de dados e introduzem variações realistas.

### Transformações nos conjuntos de validação e teste

- `Resize((224, 224))`
- `ToTensor()`
- Normalização padrão do ImageNet

Aqui não há transformações aleatórias para garantir reprodutibilidade.


In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),  # Resize to slightly larger first
    transforms.RandomCrop(224),     # Random crop for data augmentation

    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],  # ImageNet normalization
                        std=[0.229, 0.224, 0.225])
])

# Test/validation transforms (no augmentation)
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

print("Data transforms created successfully!")


## 5.4 Arquitetura do Modelo: EfficientNet-B0

O modelo utilizado é uma **EfficientNet-B0** importada do `torchvision.models` com pesos pré-treinados no ImageNet.

### Ajustes implementados no código

- Camadas iniciais congeladas (`requires_grad = False`)
- Apenas as últimas camadas convolucionais e o classificador são treináveis
- A camada final é substituída por:

``nn.Linear(in_features, 1)``

- A saída escalar alimenta a função sigmoide para classificação binária

Com isso, a rede torna-se um classificador melanoma vs. não-melanoma.


In [ ]:
# Transformações de Dados e Data Augmentation

# Transformações para treino (com augmentação)
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Transformações para validação e teste (sem augmentação)
test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("Transformações configuradas:")
print(f"Train transforms: {train_transforms}")
print(f"Test transforms: {test_transforms}")

In [ ]:
# Criação de Datasets e DataLoaders

# Criar datasets
train_dataset = datasets.ImageFolder(root=train_dir, transform=train_transforms)
test_dataset = datasets.ImageFolder(root=test_dir, transform=test_transforms)
val_dataset = datasets.ImageFolder(root=val_dir, transform=test_transforms)

# Obter nomes das classes
class_names = train_dataset.classes
print(f"Classes: {class_names}")

# Configurações do DataLoader
BATCH_SIZE = 32
NUM_WORKERS = 4

# Criar DataLoaders
train_dataloader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_dataloader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print(f"\nDataLoaders criados:")
print(f"Train batches: {len(train_dataloader)}")
print(f"Test batches: {len(test_dataloader)}")
print(f"Validation batches: {len(val_dataloader)}")

In [ ]:
# Definição do Modelo: EfficientNet-B0

# Carregar modelo pré-treinado
weights = EfficientNet_B0_Weights.DEFAULT
model = models.efficientnet_b0(weights=weights)

# Congelar camadas base (opcional - pode ser ajustado)
for param in model.features.parameters():
    param.requires_grad = False

# Modificar a camada de classificação para saída binária
num_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.2, inplace=True),
    nn.Linear(num_features, 1)  # Saída binária (logit único)
)

# Mover modelo para o dispositivo
model = model.to(device)

print("Modelo configurado:")
print(f"Arquitetura: EfficientNet-B0")
print(f"Camada de classificação modificada para saída binária")
print(f"Dispositivo: {device}")

In [ ]:
# Configuração de Treinamento

# Função de perda: BCEWithLogitsLoss (combina Sigmoid + BCE)
loss_fn = nn.BCEWithLogitsLoss()

# Otimizador: Adam
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Learning Rate Scheduler: ReduceLROnPlateau
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=3
)

print("Configuração de treinamento completa!")
print(f"Loss function: {loss_fn}")
print(f"Optimizer: {optimizer}")
print(f"Learning rate scheduler: {scheduler}")

# 6. Resultados Experimentais

## Funções de Treinamento e Validação

Implementamos funções customizadas para treinamento e validação que calculam não apenas a perda e acurácia, mas também o **F1-Score** para a classe Melanoma, métrica crítica para este problema.

## Código Python: Treinamento e Avaliação


In [ ]:
# Funções de Treinamento e Validação

def train_step(model: torch.nn.Module,
               data_loader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               optimizer: torch.optim.Optimizer,
               device: torch.device = device):
    model.train()
    train_loss, train_acc = 0, 0

    progress_bar = tqdm(data_loader, desc="Training", leave=False)

    for batch, (X, y) in enumerate(progress_bar):
        X, y = X.to(device, non_blocking=True), y.to(device, non_blocking=True)
        y_float = y.float().unsqueeze(1)

        logits = model(X)
        loss = loss_fn(logits, y_float)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).long()

        acc = accuracy_fn(y_true=y, y_pred=preds.squeeze(1))

        train_loss += loss.item()
        train_acc += acc

        progress_bar.set_postfix({
            'batch_loss': f'{loss.item():.4f}',
            'batch_acc': f'{acc:.2f}%'
        })

    train_loss /= len(data_loader)
    train_acc /= len(data_loader)

    return train_loss, train_acc


def val_step(model: torch.nn.Module,
             data_loader: torch.utils.data.DataLoader,
             loss_fn: torch.nn.Module,
             device: torch.device = device):
    """
    Passo de validação retornando loss, acc e F1 para a classe positiva (melanoma = 1).
    """
    model.eval()
    val_loss, val_acc = 0, 0
    all_preds = []
    all_labels = []

    progress_bar = tqdm(data_loader, desc="Validation", leave=False)

    with torch.inference_mode():
        for batch, (X, y) in enumerate(progress_bar):
            X, y = X.to(device, non_blocking=True), y.to(device, non_blocking=True)

            y_float = y.float().unsqueeze(1)
            logits = model(X)
            loss = loss_fn(logits, y_float)

            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).long()

            acc = accuracy_fn(y_true=y, y_pred=preds.squeeze(1))

            val_loss += loss.item()
            val_acc += acc

            all_labels.extend(y.cpu().tolist())
            all_preds.extend(preds.squeeze(1).cpu().tolist())

            progress_bar.set_postfix({
                'batch_loss': f'{loss.item():.4f}',
                'batch_acc': f'{acc:.2f}%'
            })

    val_loss /= len(data_loader)
    val_acc /= len(data_loader)

    # F1 para classe positiva (assumindo melanoma = 1)
    val_f1 = f1_score(all_labels, all_preds, pos_label=1)

    return val_loss, val_acc, val_f1

In [ ]:
# Função de Treinamento Principal com Early Stopping

def train(model: torch.nn.Module,
          train_dataloader: torch.utils.data.DataLoader,
          val_dataloader: torch.utils.data.DataLoader,
          optimizer: torch.optim.Optimizer,
          loss_fn: torch.nn.Module,
          scheduler: torch.optim.lr_scheduler.ReduceLROnPlateau,
          epochs: int,
          device: torch.device = device,
          patience: int = 10,
          min_delta: float = 1e-4,
          best_model_path: str = "best_model_f1.pth"):
    """
    Treina e valida o modelo, com early stopping baseado em F1 (classe melanoma = 1).

    - Para quando 'patience' épocas passarem sem aumento significativo de F1.
    - Salva o melhor modelo (maior F1) em best_model_path.
    """

    results = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
        "val_f1": []
    }

    best_val_f1 = 0.0
    best_val_acc = 0.0
    epochs_sem_melhora = 0

    for epoch in tqdm(range(epochs), desc="Epochs"):
        train_loss, train_acc = train_step(
            model=model,
            data_loader=train_dataloader,
            loss_fn=loss_fn,
            optimizer=optimizer,
            device=device
        )

        val_loss, val_acc, val_f1 = val_step(
            model=model,
            data_loader=val_dataloader,
            loss_fn=loss_fn,
            device=device
        )

        results["train_loss"].append(train_loss)
        results["train_acc"].append(train_acc)
        results["val_loss"].append(val_loss)
        results["val_acc"].append(val_acc)
        results["val_f1"].append(val_f1)

        print(
            f"Epoch: {epoch+1} | "
            f"train_loss: {train_loss:.4f} | "
            f"train_acc: {train_acc:.2f}% | "
            f"val_loss: {val_loss:.4f} | "
            f"val_acc: {val_acc:.2f}% | "
            f"val_f1 (melanoma): {val_f1:.4f}"
        )

        scheduler.step(val_loss)

        if val_f1 - best_val_f1 > min_delta:
            best_val_f1 = val_f1
            best_val_acc = val_acc
            epochs_sem_melhora = 0

            torch.save(model.state_dict(), best_model_path)
            print(f"⬆️ Novo melhor F1 de validação: {best_val_f1:.4f} | modelo salvo em {best_model_path}")
        else:
            epochs_sem_melhora += 1
            print(f"Sem melhora significativa em F1 há {epochs_sem_melhora} época(s).")

            if epochs_sem_melhora >= patience:
                print(f"⏹️ Early stopping acionado na época {epoch+1} (paciente = {patience}).")
                break

    print(f"Melhor F1 de validação (melanoma): {best_val_f1:.4f}")
    print(f"Melhor acurácia de validação correspondente: {best_val_acc:.2f}%")

    return results, best_val_acc, best_val_f1

In [ ]:
# Execução do Treinamento

EPOCHS = 50
BEST_MODEL_PATH = "best_model_f1.pth"

start_time = timer()

model_results, best_val_acc, best_val_f1 = train(
    model=model,
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    optimizer=optimizer,
    loss_fn=loss_fn,
    scheduler=scheduler,
    epochs=EPOCHS,
    device=device,
    patience=10,
    min_delta=1e-4,
    best_model_path=BEST_MODEL_PATH
)

end_time = timer()
total_train_time = end_time - start_time

print(f"\nTotal training time: {total_train_time:.2f} seconds")
print(f"Best Validation Accuracy: {best_val_acc:.2f}%")
print(f"Best Validation F1 (melanoma): {best_val_f1:.4f}")

# Carregar o melhor modelo
print(f"\nCarregando o melhor modelo salvo em {BEST_MODEL_PATH} para avaliação final...")
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
model.to(device)

In [ ]:
# Função de Teste do Modelo

def test_model(model: torch.nn.Module,
               data_loader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               device: torch.device = device):
    """Tests a model on a test dataset."""
    model.eval()
    test_loss, test_acc = 0, 0
    all_preds = []
    all_labels = []

    progress_bar = tqdm(data_loader, desc="Testing", leave=False)

    with torch.inference_mode():
        for X, y in progress_bar:
            X, y = X.to(device), y.to(device)

            y_logits = model(X)
            y_float = y.float().unsqueeze(1)
            loss = loss_fn(y_logits, y_float)
            test_loss += loss.item()

            y_pred_prob = torch.sigmoid(y_logits).squeeze(1)
            y_pred_class = (y_pred_prob > 0.5).long()

            test_acc += accuracy_fn(y_true=y.cpu(), y_pred=y_pred_class.cpu())

            all_preds.extend(y_pred_class.cpu().tolist())
            all_labels.extend(y.cpu().tolist())

    test_loss = test_loss / len(data_loader)
    test_acc = test_acc / len(data_loader)

    return {
        "model_name": model.__class__.__name__,
        "model_loss": test_loss,
        "model_acc": test_acc,
        "predictions": all_preds,
        "labels": all_labels
    }

# Executar teste
test_results = test_model(
    model=model,
    data_loader=test_dataloader,
    loss_fn=loss_fn,
    device=device
)

print(f"\n=== Resultados do Teste ===")
print(f"Test Loss: {test_results['model_loss']:.4f}")
print(f"Test Accuracy: {test_results['model_acc']:.2f}%")

In [ ]:
# Métricas Detalhadas de Teste

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import seaborn as sns

# Classification Report
print("\n=== Classification Report ===")
print(classification_report(
    test_results['labels'],
    test_results['predictions'],
    target_names=class_names
))

# Confusion Matrix
cm = confusion_matrix(test_results['labels'], test_results['predictions'])
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

# F1-Score para Melanoma
test_f1 = f1_score(test_results['labels'], test_results['predictions'], pos_label=1)
print(f"\nF1-Score (Melanoma): {test_f1:.4f}")

In [ ]:
# Visualização das Curvas de Treinamento

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(model_results['train_loss'], label='Train Loss', color='#2d5f5d')
axes[0].plot(model_results['val_loss'], label='Val Loss', color='#8ab8b6')
axes[0].set_title('Loss ao Longo das Épocas')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(model_results['train_acc'], label='Train Acc', color='#2d5f5d')
axes[1].plot(model_results['val_acc'], label='Val Acc', color='#8ab8b6')
axes[1].set_title('Acurácia ao Longo das Épocas')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Acurácia (%)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# F1-Score
axes[2].plot(model_results['val_f1'], label='Val F1 (Melanoma)', color='#d4a373')
axes[2].set_title('F1-Score (Melanoma) ao Longo das Épocas')
axes[2].set_xlabel('Época')
axes[2].set_ylabel('F1-Score')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 7. Discussão e Limitações

## Decisões Essenciais para a Validação

O sucesso do modelo de triagem depende fundamentalmente de duas estratégias:

### 1. Mitigação do Desbalanceamento

O dataset HAM10000 apresenta um desbalanceamento severo (11% melanoma vs 89% não-melanoma). Para endereçar este desafio, implementamos:

- **Class Weights:** Penalização maior para erros na classe minoritária
- **Focal Loss:** Foco em exemplos difíceis de classificar
- **Técnicas de Oversampling/Augmentação:** Aumento artificial da representação da classe melanoma

### 2. Prevenção de Data Leakage

O campo **lesion_id** indica que múltiplas imagens podem pertencer à mesma lesão. Para garantir validação realista:

- **Split Estratificado:** Agrupamento por lesion_id
- **Separação Rigorosa:** Garantia de que imagens da mesma lesão não apareçam em treino e teste simultaneamente
- **Validação com Stratified Group K-Fold:** Técnica robusta para avaliação

## Recursos e Desafios (Lições da EDA)

### Dataset HAM10000: Adequado mas Desbalanceado

**Pontos Fortes:**
- Grande volume de imagens (10.015)
- Metadados clínicos ricos (idade, sexo, localização)
- Amplamente utilizado como benchmark
- Licença adequada para uso acadêmico

**Desafios:**
- **Extremamente desbalanceado** na classificação binária
- Risco de **leakage** devido ao lesion_id
- Variação de qualidade de imagem exige pré-processamento rigoroso

### Variação de Imagem e Pré-processamento

A heterogeneidade do dataset (múltiplas fontes) exige pipeline robusto:

- **Normalização de cor:** Shades-of-Gray
- **Remoção de artefatos:** Inpainting para pelos e reflexos
- **Padronização:** Resize/crop consistente

### Viés de Representação

A maioria dos repositórios de dermatoscopia tem **baixa representação de tons de pele escuros**. Questões de **fairness** e **generalização** devem ser consideradas em trabalhos futuros.

## Métricas-Chave: Sensibilidade e AUC

Dado o contexto clínico (detecção de câncer), as métricas prioritárias são:

- **Sensibilidade (Recall) ≥ 0,90:** Minimizar falsos negativos (melanomas não detectados)
- **AUC-ROC ≥ 0,90:** Capacidade geral de discriminação
- **Especificidade ≥ 0,75:** Controlar falsos positivos (encaminhamentos desnecessários)

## Limitações do Estudo

### 1. Limitações do Dataset

- **Desbalanceamento severo** pode comprometer a sensibilidade mesmo com técnicas de mitigação
- **Viés de representação:** Baixa diversidade de tons de pele
- **Qualidade heterogênea:** Variação significativa entre fontes

### 2. Limitações Técnicas

- **Restrições de hardware:** Modelo deve ser leve (<5MB) para inferência on-device
- **Trade-off precisão vs eficiência:** Modelos mais leves podem ter performance inferior
- **Generalização:** Performance em dados reais (fora do HAM10000) precisa ser validada

### 3. Limitações Clínicas

- **Não substitui diagnóstico médico:** Ferramenta de triagem, não diagnóstico definitivo
- **Contexto clínico limitado:** Modelo não considera histórico do paciente
- **Validação clínica necessária:** Estudos prospectivos são essenciais antes de uso real

## Próximos Passos

Para endereçar essas limitações, os próximos passos incluem:

1. **Validação em datasets externos** com maior diversidade
2. **Incorporação de metadados clínicos** (modelo multimodal)
3. **Estudos de fairness** para garantir equidade entre diferentes tons de pele
4. **Validação clínica prospectiva** em ambiente real


# 8. Conclusão e Trabalhos Futuros

## Conclusão

Este projeto demonstrou a viabilidade de desenvolver uma ferramenta de **classificação binária de melanoma** utilizando técnicas de **Deep Learning** e **Visão Computacional** aplicadas ao dataset HAM10000.

### Principais Conquistas

1. **Pipeline Completo:** Desenvolvimento de pipeline end-to-end desde pré-processamento até inferência
2. **Modelo Eficiente:** Implementação de arquitetura leve (EfficientNet-B0) adequada para inferência on-device
3. **Métricas Robustas:** Foco em sensibilidade e F1-Score para a classe melanoma
4. **Early Stopping Inteligente:** Prevenção de overfitting com monitoramento de F1-Score

### Impacto Potencial

A ferramenta desenvolvida tem potencial para:

- **Democratizar o acesso** à triagem dermatológica em regiões com escassez de especialistas
- **Reduzir o tempo de espera** para consultas especializadas através de priorização de casos de alto risco
- **Padronizar a avaliação inicial** de lesões suspeitas
- **Facilitar o monitoramento domiciliar** de pacientes de risco

## Trabalhos Futuros

### Próximo Ciclo de Desenvolvimento

1. **Limpeza de dados e Split agrupado (lesion_id)**
   - Implementação rigorosa de split estratificado
   - Prevenção completa de data leakage

2. **Treinamento da Baseline (MobileNet/EfficientNet)**
   - Experimentação com diferentes arquiteturas
   - Otimização de hiperparâmetros

3. **Desenvolvimento do Protótipo Mobile (quantização e UI)**
   - Quantização do modelo para redução de tamanho
   - Desenvolvimento de interface de usuário intuitiva

4. **Estudo e Plano de Teste com imagens reais (ética)**
   - Validação em dados externos
   - Considerações éticas e regulatórias

### Melhorias Técnicas

- **Modelos Multimodais:** Incorporação de metadados clínicos (idade, sexo, localização)
- **Ensemble Methods:** Combinação de múltiplos modelos para maior robustez
- **Técnicas Avançadas de Augmentação:** Uso de GANs para geração sintética de melanomas
- **Explicabilidade Aprimorada:** Implementação de Grad-CAM++ e outras técnicas de interpretabilidade

### Validação Clínica

- **Estudos Prospectivos:** Validação em ambiente clínico real
- **Análise de Fairness:** Avaliação de performance em diferentes tons de pele
- **Feedback de Especialistas:** Incorporação de insights de dermatologistas

### Considerações Regulatórias

- **Certificação Médica:** Adequação a normas regulatórias (ANVISA, FDA)
- **Privacidade de Dados:** Conformidade com LGPD/GDPR
- **Responsabilidade Ética:** Diretrizes claras sobre uso e limitações da ferramenta

## Mensagem Final

O desenvolvimento de ferramentas de IA para auxílio diagnóstico representa uma oportunidade única de democratizar o acesso à saúde de qualidade. No entanto, é fundamental manter o rigor científico, a transparência sobre limitações e o compromisso ético com a segurança dos pacientes.

Este projeto estabelece uma base sólida para futuros desenvolvimentos, mas reconhece que o caminho até a implementação clínica real requer validação extensiva, colaboração interdisciplinar e compromisso contínuo com a excelência.


# 9. Referências

## Datasets

1. **HAM10000 Dataset**
   - Tschandl, P., Rosendahl, C., & Kittler, H. (2018). The HAM10000 dataset, a large collection of multi-source dermatoscopic images of common pigmented skin lesions. *Scientific Data*, 5, 180161.
   - Disponível em: [ISIC Archive](https://www.isic-archive.com/) e [Harvard Dataverse](https://dataverse.harvard.edu/)

## Estatísticas de Saúde

2. **GLOBOCAN 2022**
   - Global Cancer Observatory. (2022). Cancer Today: Data visualization tools for exploring the global cancer burden in 2020.
   - Disponível em: [https://gco.iarc.fr/](https://gco.iarc.fr/)

3. **SEER (Surveillance, Epidemiology, and End Results Program)**
   - National Cancer Institute. Melanoma of the Skin - Cancer Stat Facts.
   - Disponível em: [https://seer.cancer.gov/](https://seer.cancer.gov/)

## Arquiteturas de Deep Learning

4. **EfficientNet**
   - Tan, M., & Le, Q. V. (2019). EfficientNet: Rethinking Model Scaling for Convolutional Neural Networks. *International Conference on Machine Learning (ICML)*.

5. **MobileNetV3**
   - Howard, A., Sandler, M., Chu, G., et al. (2019). Searching for MobileNetV3. *International Conference on Computer Vision (ICCV)*.

## Técnicas de Treinamento

6. **Focal Loss**
   - Lin, T. Y., Goyal, P., Girshick, R., He, K., & Dollár, P. (2017). Focal Loss for Dense Object Detection. *IEEE Transactions on Pattern Analysis and Machine Intelligence*.

7. **Class Imbalance**
   - Buda, M., Maki, A., & Mazurowski, M. A. (2018). A systematic study of the class imbalance problem in convolutional neural networks. *Neural Networks*, 106, 249-259.

## Explicabilidade

8. **Grad-CAM**
   - Selvaraju, R. R., Cogswell, M., Das, A., et al. (2017). Grad-CAM: Visual Explanations from Deep Networks via Gradient-based Localization. *International Conference on Computer Vision (ICCV)*.

## Recursos Educacionais

9. **PyTorch Deep Learning**
   - Bourke, D. Learn PyTorch for Deep Learning: Zero to Mastery.
   - Disponível em: [https://www.learnpytorch.io/](https://www.learnpytorch.io/)
   - Helper Functions: [GitHub Repository](https://github.com/mrdbourke/pytorch-deep-learning)

## Frameworks e Bibliotecas

10. **PyTorch**
    - Paszke, A., Gross, S., Massa, F., et al. (2019). PyTorch: An Imperative Style, High-Performance Deep Learning Library. *Advances in Neural Information Processing Systems (NeurIPS)*.

11. **scikit-learn**
    - Pedregosa, F., Varoquaux, G., Gramfort, A., et al. (2011). Scikit-learn: Machine Learning in Python. *Journal of Machine Learning Research*, 12, 2825-2830.

## Licenças e Ética

12. **Creative Commons**
    - Creative Commons BY-NC 4.0 License
    - Disponível em: [https://creativecommons.org/licenses/by-nc/4.0/](https://creativecommons.org/licenses/by-nc/4.0/)

---

**Nota:** Este notebook foi desenvolvido como parte do Projeto de Visão Computacional - Tarefa 1, Grupo 18.

**Autores:** Enzo Souza, Vinicius Hardy, Angelo Pereira

**Data:** Novembro de 2025

**Instituição:** [Nome da Instituição - se aplicável]
